### Part 1 write-up
1. Not injective: This tokenizer converts string to lower case before creating tokens hence “Anubhav” and “anubhav” will get the same tokens even though the string is different.
2. Not invertible: This tokenizer reduces multiple whitespaces to single whitespace hence the reconstructed string will never be the same as the starting string.
3. Not preserving concatenation:  The tokenizer uses Wordpiece algorithm which processes “Anu”, “bhav” separately than “Anubhav”. Hence it doesn’t preserve the concatenation property

### Part 2

In [1]:
import re
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm 
from time import sleep
from sklearn.metrics import mean_absolute_error
from huggingface_hub import notebook_login

In [2]:
# Helper function for debugging
def dprint(s, debug):
    if debug:
        print(s)

In [3]:
# --- Check Device ---
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from huggingface_hub import login, whoami

login(token="PUT_YOUR_TOKEN", add_to_git_credential=False)
print(whoami()["name"])

ap84pd


In [5]:
# # --- Model Loading ---
# # We'll run our models using Hugging Face's transformers library on HPC/ Google Colab/ Lightning.ai
# # The Llama models are gated, meaning you must request access on their Hugging Face pages.
# # Once you have access, you need to log in here to download the model weights.

# # Run this command in your terminal when you are running this notebook for the 1st time
# # git config --global credential.helper store

# print("Please log in to your Hugging Face account.")
# notebook_login()

In [7]:
# This will be our primary model for most of the assignment.
model_id_1 = "meta-llama/Llama-2-7b-chat-hf"

In [8]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [9]:
def call_model(prompt, student_configs, post_processing_fn, model_obj, tokenizer_obj, debug=False):
    """
    Generates a response using the provided local Hugging Face model and tokenizer.
    """
    # 1. Tokenize the input prompt
    inputs = tokenizer_obj(prompt, return_tensors="pt").to(device)

    hf_configs = student_configs.copy()
    if 'max_tokens' in hf_configs:
        # `generate` uses `max_new_tokens` to specify the length of the output
        hf_configs['max_new_tokens'] = hf_configs.pop('max_tokens')
    if 'stop' in hf_configs:
        del hf_configs['stop'] # Stop sequences are handled differently; we'll ignore for simplicity

    # 2. Generate output tokens
    outputs = model_obj.generate(**inputs, **hf_configs).to(device)
    
    # 3. Decode the generated tokens back to a string
    # We slice the output to only get the newly generated text, not the original prompt
    result_new = tokenizer_obj.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    dprint("************ Prompt ************", debug)
    dprint(prompt, debug)
    dprint("\n************ Raw Response ************", debug)
    dprint(result_new, debug)

    # 4. Apply post-processing to extract the final answer
    final_output = post_processing_fn(result_new)
    
    dprint("\n************ Final Output ************", debug)
    dprint(final_output, debug)

    return final_output

In [10]:
def get_addition_pairs(lower_bound, upper_bound, rng):
    """Generates two random integers within a specified range."""
    int_a = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    int_b = int(np.ceil(rng.uniform(lower_bound, upper_bound)))
    return int_a, int_b

def test_range(added_prompt, prompt_configs, rng, model_obj, tokenizer_obj, n_sample=30,
               lower_bound=1, upper_bound=10, fixed_pairs=None,
               pre_processing=lambda x:x, post_processing=lambda y:y,
               debug=False):
    """
    Tests a language model's addition performance over a range of numbers.

    Args:
        added_prompt (tuple): A tuple containing the prefix and suffix for the prompt.
        prompt_configs (dict): Configuration parameters for the model's generate function.
        rng (numpy.random.Generator): A random number generator instance.
        model_obj (transformers.PreTrainedModel): The loaded Hugging Face model object.
        tokenizer_obj (transformers.PreTrainedTokenizer): The loaded Hugging Face tokenizer object.
        n_sample (int): The number of random pairs to generate if fixed_pairs is None.
        lower_bound (int): The lower bound for number generation.
        upper_bound (int): The upper bound for number generation.
        fixed_pairs (list, optional): A list of specific integer tuples to test.
        pre_processing (function): A function to apply to the input string before prompting.
        post_processing (function): A function to extract the integer answer from the model's output.
        debug (bool): If True, prints detailed debugging information.

    Returns:
        dict: A dictionary containing performance metrics (res, acc, mae, prompt_length).
    """
    # --- Lists for storing results ---
    int_as = []
    int_bs = []
    answers = []
    model_responses = []
    correct = []
    prompts = []
    
    # --- Determine the test cases ---
    iterations = range(n_sample) if fixed_pairs is None else fixed_pairs
    
    for v in tqdm(iterations):
        if fixed_pairs is None:
            # Generate two new numbers if no fixed pairs are provided
            int_a, int_b = get_addition_pairs(lower_bound=lower_bound, upper_bound=upper_bound, rng=rng)
        else:
            # Use the provided fixed pairs
            int_a, int_b = v
            
        # --- Construct the prompt for two numbers ---
        fixed_prompt = f'{int_a}+{int_b}'
        fixed_prompt = pre_processing(fixed_prompt)
        
        prefix, suffix = added_prompt
        prompt = prefix + fixed_prompt + suffix
        
        # --- Get the model's response ---
        model_response = call_model(prompt, prompt_configs, post_processing, model_obj, tokenizer_obj, debug=debug)
        
        # --- Calculate the correct answer for two numbers ---
        answer = int_a + int_b
        
        # --- Append all results for analysis ---
        int_as.append(int_a)
        int_bs.append(int_b)
        prompts.append(prompt)
        answers.append(answer)
        model_responses.append(model_response)
        correct.append((answer == model_response))
        sleep(0.1)

    # --- Create a DataFrame to display the results for two numbers ---
    df = pd.DataFrame({
        'int_a': int_as, 
        'int_b': int_bs, 
        'prompt': prompts, 
        'answer': answers, 
        'response': model_responses, 
        'correct': correct
    })
    print(df)
    
    # --- Calculate and return performance metrics ---
    mae = mean_absolute_error(df['answer'], df['response'])
    acc = df.correct.sum() / len(df)
    prompt_length = len(prefix) + len(suffix)
    res = acc * (1 / prompt_length) * (1 - mae / (1 * 10**4))
    
    return {'res': res, 'acc': acc, 'mae': mae, 'prompt_length': prompt_length}

###  Part 1. Zero Shot Addition

**Example: Zero-shot single-digit addition**

In [11]:
# All of this remains the same
added_prompt = ('Question: What is ', '?\\nAnswer: ')
prompt_config = {'max_tokens': 2,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}

def your_pre_processing(input_string):
    return input_string

def your_post_processing(output_string):
    only_digits = re.sub(r"\D", "", output_string)
    try:
        res = int(only_digits)
    except:
        res = 0
    return res

# The model name string is no longer passed to the function
# It was used in the previous cell to load the 'model' and 'tokenizer' objects
print(f"Testing model: {model_id_1}")
seed = 0
rng = np.random.default_rng(seed)

# This is the only line that changes
res = test_range(
    added_prompt=added_prompt,
    prompt_configs=prompt_config,
    rng=rng,
    model_obj=model, 
    tokenizer_obj=tokenizer,
    n_sample=10,
    lower_bound=1,
    upper_bound=10,
    fixed_pairs=None,
    pre_processing=your_pre_processing,
    post_processing=your_post_processing,
    debug=False
)
print(res)

Testing model: meta-llama/Llama-2-7b-chat-hf


  0%|          | 0/10 [00:00<?, ?it/s]

   int_a  int_b                             prompt  answer  response  correct
0      7      4   Question: What is 7+4?\nAnswer:       11        11     True
1      2      2   Question: What is 2+2?\nAnswer:        4         4     True
2      9     10  Question: What is 9+10?\nAnswer:       19        19     True
3      7      8   Question: What is 7+8?\nAnswer:       15        15     True
4      6     10  Question: What is 6+10?\nAnswer:       16        16     True
5      9      2   Question: What is 9+2?\nAnswer:       11        11     True
6      9      2   Question: What is 9+2?\nAnswer:       11        11     True
7      8      3   Question: What is 8+3?\nAnswer:       11        11     True
8      9      6   Question: What is 9+6?\nAnswer:       15        15     True
9      4      5   Question: What is 4+5?\nAnswer:        9         9     True
{'res': 0.034482758620689655, 'acc': 1.0, 'mae': 0.0, 'prompt_length': 29}


**Example: Zero-shot 7-digit addition**

In [12]:

prompt_config['max_tokens'] = 8
rng = np.random.default_rng(seed)

# The call to test_range is updated to pass the model and tokenizer objects.
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10154736    False  
1   2517511     True  
2  17535200    False  
3  14026688    False  
4   5892625    

-----------

**Q3a.** In your opinion, what are some factors that cause language model performance to deteriorate from 1 digit to 7 digits?

Answer:
- There are 10 x 10 possible single digit addition pairs possible whereas in case of 7 digits sample space is huge. So model will have to learn to perform the addition rather than just memorizing it.
- Adding 7 digits requires many more arithmetic steps compared to 1 digit and the carry also must be propagated correctly till the end

-----------

**Q3b**. Play around with the config parameters ('max_tokens','temperature','top_k','top_p','repetition_penalty')
* What does each parameter represent?
* How does increasing each parameter change the generation?

Answer: 
- max_tokens: maximum number of tokens generated in the output. If we increase this then model will be able to generate for longer
- temperature: adds a variability in the output by rescaling the logit values. Higher temperature makes output less deterministic
- top_k: next generated token will be chosen out of "k" top choices
- top_p: limits the model to the smallest set of tokens whose cumulative probability reaches "p"
- repetition_penalty: penalizes the token that model has already generated. Value = 1 means no penalty

**Q3c**. Do 7-digit addition with Qwen3 8B.

* How does the performance change?
* What are some factors that cause this change?

In [13]:
# --- Before loading Qwen 3, offload Llama 2 to free up VRAM ---

# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model
del tokenizer

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print("Llama 2 model offloaded and GPU memory cleared.")

Llama 2 model offloaded and GPU memory cleared.


In [14]:
# --- Load Qwen 3 8B ---
# This is a different model, so we need to load its specific tokenizer and weights.
model_id_2 = "Qwen/Qwen3-8B"

print(f"\nLoading tokenizer for {model_id_2}...")
tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)

print(f"Loading model: {model_id_2}...")
model_2 = AutoModelForCausalLM.from_pretrained(
    model_id_2,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_2} model loaded successfully!")


Loading tokenizer for Qwen/Qwen3-8B...
Loading model: Qwen/Qwen3-8B...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Qwen/Qwen3-8B model loaded successfully!


In [15]:
# --- Test on 7-digit addition ---
prompt_config['max_tokens'] = 8

print(prompt_config)
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

{'max_tokens': 8, 'temperature': 0.7, 'top_k': 50, 'top_p': 0.6, 'repetition_penalty': 1, 'stop': []}


  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

   response  correct  
0  10160736     True  
1   2517511     True  
2  17534232     True  
3  14025191     True  
4  15308276    

-----------

Answer: Qwen 3-8B model did much better than Llama2 model. Accuracy of Qwen3 is 80% whereas for Llama is 20%. Qwen3 is a newer model than Llama2 with more math problems in dataset. Additionally Qwen3 is instruction tuned model whereas Llama2 is base model. This is the biggest factor for Qwen3 performing better than Llama2

-----------

**Q3d.** Here we're giving our language model the prior that the sum of two 7-digit numbers must have a maximum of 8 digits. (by setting max_token=8). What if we remove this prior by increasing the max_token to 20? 
* Does the model perform well?
* What are some reasons why?

Answer:
- Increasing max_tokens has actually resulted in degradation of performance from 80% to 50%.
- For example the first example was correct in case of max_tokens=8 but incorrect in case of 20. It means that model just got lucky due to truncation in the output.
- Model doesn't know actually when to stop and keeps generating output. Hence accuracy alone was not a good metric as it's sensitive to max_tokens

In [16]:

prompt_config['max_tokens'] = 20
rng = np.random.default_rng(seed)
res_2 = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model_2,              # Pass the loaded model object
    tokenizer_obj=tokenizer_2,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res_2)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                        prompt    answer  \
0  6732655  3428081  Question: What is 6732655+3428081?\nAnswer:   10160736   
1  1368762  1148749  Question: What is 1368762+1148749?\nAnswer:    2517511   
2  8319432  9214800  Question: What is 8319432+9214800?\nAnswer:   17534232   
3  6459722  7565469  Question: What is 6459722+7565469?\nAnswer:   14025191   
4  5892625  9415651  Question: What is 5892625+9415651?\nAnswer:   15308276   
5  8342682  1024647  Question: What is 8342682+1024647?\nAnswer:    9367329   
6  8716638  1302271  Question: What is 8716638+1302271?\nAnswer:   10018909   
7  7566899  2580901  Question: What is 7566899+2580901?\nAnswer:   10147800   
8  8768610  5873151  Question: What is 8768610+5873151?\nAnswer:   14641761   
9  3697407  4804185  Question: What is 3697407+4804185?\nAnswer:    8501592   

         response  correct  
0  10160736723586    False  
1  25175111468762    False  
2        17534232     True  
3        14025

In [17]:
# 1. Delete the model and tokenizer variables from memory.
# Replace 'model' and 'tokenizer' with the actual variable names you used for Llama 2.
del model_2
del tokenizer_2

# 2. Run Python's garbage collector and empty PyTorch's CUDA cache.
# This is the crucial step to actually release the GPU memory.
gc.collect()
torch.cuda.empty_cache()

print(f"{model_id_2} offloaded and GPU memory cleared.")

Qwen/Qwen3-8B offloaded and GPU memory cleared.


### Part 2. In Context Learning

We will try to improve the performance of 7-digit addition via in-context learning.
We will use [llama-2-7b]. Below is a simple example.

In [18]:
print(f"\nLoading tokenizer for {model_id_1}...")
# The tokenizer turns our text prompt into numbers the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_id_1)

print(f"Loading model: {model_id_1}...")
# This downloads the model weights to your environment.
# torch_dtype=torch.bfloat16 uses half-precision floats to save memory.
# device_map="auto" automatically puts the model on the GPU if available.
model = AutoModelForCausalLM.from_pretrained(
    model_id_1,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print(f"{model_id_1} model loaded successfully!")


Loading tokenizer for meta-llama/Llama-2-7b-chat-hf...
Loading model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


meta-llama/Llama-2-7b-chat-hf model loaded successfully!


In [19]:

added_prompt = ('Question: What is 3+7?\nAnswer: 10\n Question: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
prompt_config = {'max_tokens': 8,
                'temperature': 0.7,
                'top_k': 50,
                'top_p': 0.6,
                'repetition_penalty': 1,
                'stop': []}
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)

print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10154739    False  
1   2517511   2517611    False  
2  17534232  17533242    False  
3  14025191   7324291    False  
4  15308276  1030816

**Q4a**.
* How does the performance change with the baseline in-context learning prompt? (compare with "Example: Zero-shot 7-digit addition" in Q1)
* What are some factors that cause this change?

Answer: 
- Adding an example in the prompt didn't improve the performance at all. In fact previously it was 20% now 0%.
- We provided 1 digit addition in the example but we are asking the model to perform 7 digit addition which can be a reason for such performance.
- This also shows the model's inability of performing arithmetic operations.

-----------

Now we will remove the prior on output length and re-evaluate the performance of our baseline one-shot learning prompt. We need to modify our post processing function to extract the answer from the output sequence. In this case, it is the number in the first line that starts with "Answer: ".

**Q4b**.
* Modify the post processing function
* How does the performance change when we relax the output length constraint? (compare with Q4a)
* What are some factors that cause this change?

Answer: 
- Increased the max_tokens to 50 and limited the output to only 1st line. The accuracy didn't improve at all. Infact the MAE has increased which also aligns with the fact that with increase in tokens the chances of model making errors increases.
- Longer generation doesn't improve the model's arithmetic ability.

In [20]:
#Write your updated post processing function here
def your_post_processing(output_string):
    first_line = output_string.splitlines()[0] if output_string.splitlines() else ""
    first_line = re.sub(r'\s*Answer:\s*', '', first_line, flags=re.IGNORECASE)
    match = re.search(r'\d+', first_line)
    return int(match.group()) if match else 0

In [21]:
prompt_config['max_tokens'] = 50 # changed from 8, assuming we don't know the output length
                
rng = np.random.default_rng(seed)
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                             prompt  \
0  6732655  3428081  Question: What is 3+7?\nAnswer: 10\n Question:...   
1  1368762  1148749  Question: What is 3+7?\nAnswer: 10\n Question:...   
2  8319432  9214800  Question: What is 3+7?\nAnswer: 10\n Question:...   
3  6459722  7565469  Question: What is 3+7?\nAnswer: 10\n Question:...   
4  5892625  9415651  Question: What is 3+7?\nAnswer: 10\n Question:...   
5  8342682  1024647  Question: What is 3+7?\nAnswer: 10\n Question:...   
6  8716638  1302271  Question: What is 3+7?\nAnswer: 10\n Question:...   
7  7566899  2580901  Question: What is 3+7?\nAnswer: 10\n Question:...   
8  8768610  5873151  Question: What is 3+7?\nAnswer: 10\n Question:...   
9  3697407  4804185  Question: What is 3+7?\nAnswer: 10\n Question:...   

     answer  response  correct  
0  10160736  10150442    False  
1   2517511   2517611    False  
2  17534232  17530200    False  
3  14025191   7324685    False  
4  15308276  1030816

-----------

**Q4c.** Let's change our one-shot learning example to something more "in-distribution". Previously we were using 1-digit addition as an example. Let's change it to 7-digit addition (1234567+1234567=2469134). 
* Evaluate the performance with max_tokens = 8.
* Evaluate the performance with max_tokens = 50.
* How does the performance change from 1-digit example to 7-digit example?

Answer:
- Changing the example from 1 digit to 7 digit addition has increased model's performance a bit however it's still not convincing enough that the model has learned something. Because in case of max_tokens = 50 the performance again dropped to 0. However the MAE has reduced here compared to the previous cases

In [22]:

prompt_config['max_tokens'] = 8 
added_prompt = ('Question: What is 1234567+1234567?\nAnswer: 2469134\nQuestion: What is ', '?\nAnswer: ') # Question: What is a+b?\nAnswer:
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                             prompt  \
0  1254878  2118550  Question: What is 1234567+1234567?\nAnswer: 24...   
1  7035620  6824705  Question: What is 1234567+1234567?\nAnswer: 24...   
2  6538466  4453098  Question: What is 1234567+1234567?\nAnswer: 24...   
3  9974889  9827518  Question: What is 1234567+1234567?\nAnswer: 24...   
4  7169878  6854133  Question: What is 1234567+1234567?\nAnswer: 24...   
5  7196020  4500293  Question: What is 1234567+1234567?\nAnswer: 24...   
6  2215869  7493395  Question: What is 1234567+1234567?\nAnswer: 24...   
7  5728189  3792177  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5372518  9005390  Question: What is 1234567+1234567?\nAnswer: 24...   
9  9406391  4220157  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0   3373428   3373428     True  
1  13860325  13850825    False  
2  10991564  10992064    False  
3  19802407  19802497    False  
4  14024011  1392400

In [23]:

prompt_config['max_tokens'] = 50 
res = test_range(
    added_prompt=added_prompt, 
    prompt_configs=prompt_config, 
    rng=rng, 
    model_obj=model,              # Pass the loaded model object
    tokenizer_obj=tokenizer,      # Pass the loaded tokenizer object
    n_sample=10, 
    lower_bound=1000000, 
    upper_bound=9999999, 
    fixed_pairs=None, 
    pre_processing=your_pre_processing, 
    post_processing=your_post_processing, 
    debug=False
)
print(res)

  0%|          | 0/10 [00:00<?, ?it/s]

     int_a    int_b                                             prompt  \
0  6143768  3896825  Question: What is 1234567+1234567?\nAnswer: 24...   
1  6348700  4041201  Question: What is 1234567+1234567?\nAnswer: 24...   
2  4524571  9012469  Question: What is 1234567+1234567?\nAnswer: 24...   
3  3044419  6608684  Question: What is 1234567+1234567?\nAnswer: 24...   
4  1756139  8493797  Question: What is 1234567+1234567?\nAnswer: 24...   
5  8083884  3154325  Question: What is 1234567+1234567?\nAnswer: 24...   
6  8888358  1527113  Question: What is 1234567+1234567?\nAnswer: 24...   
7  4025054  2352516  Question: What is 1234567+1234567?\nAnswer: 24...   
8  5053054  8166918  Question: What is 1234567+1234567?\nAnswer: 24...   
9  3075780  1468192  Question: What is 1234567+1234567?\nAnswer: 24...   

     answer  response  correct  
0  10040593  10034513    False  
1  10389901  10389301    False  
2  13537040   5436040    False  
3   9653103   9649004    False  
4  10249936  1025817

-----------

**Q4d.** Let's look at a specific example with large absolute error. 
* Run the cell at least 5 times. Does the error change each time? Why?
* Can you think of a prompt to reduce the error?
* Why do you think it would work?
* Does it work in practice? Why or why not?

Answer:
- Run1: {'res': -0.0, 'acc': 0.0, 'mae': 9089900.0, 'prompt_length': 1203}
- Run2: {'res': -0.0, 'acc': 0.0, 'mae': 9089900.0, 'prompt_length': 1203}
- Run3: {'res': -0.0, 'acc': 0.0, 'mae': 11717092.0, 'prompt_length': 1203}
- Run4: {'res': -0.0, 'acc': 0.0, 'mae': 11717092.0, 'prompt_length': 1203}
- Run5: {'res': -0.0, 'acc': 0.0, 'mae': 7282908.0, 'prompt_length': 1203}
1. Yes the error changes every time. This is because the model samples it's output so difference runs pick a different token. Since top_p is set to 0.6 the variation is less due t which we can see that the variation in the errors are also less
2. Prompt: split the number in the groups of 3 and start from right to left. Perform additions and track the carry as well. Finally combine all the small chunks to create the final number.
3. I though this should work as this breaks a big problem (adding 7 digits) into small problems (adding 3 digits although harder than 1 digit) to solve. I also added 2 examples to show model how to perform operations.
3. No it didn't work. The model made several mistakes from start itself. For example in all the run model identified middle as 909 and 101 in place of 090 and 010 which is wrong. Even after identifying the digits correctly the model is not computing it correct. In such cases these errors add up. The final answer depends on all the steps to be correct. Event if 1 step is incorrect it will be wrong.

In [30]:
prompt = (
    """Add the two given integers using three-digit grouping.
    Instructions:
    1. Split both numbers into groups of three digits, starting from the right.
    2. Add the corresponding groups from right to left.
    3. Include any carry from the previous group.
    4. For each group except the leftmost:
       - Write exactly three digits, padding with leading zeros if needed.
       - Carry any overflow to the next group.
    5. For the leftmost group, include the final carry without padding.
    6. Show the calculations for each group and the combined result.
    7. End with exactly:
       Final answer: <integer>

    
    Example 1:
    Question: What is 5102020 + 2304015?
    Groups: 5|102|020 and 2|304|015
    Right: 020 + 015 = 35. Write 035, carry 0.
    Middle: 102 + 304 + 0 = 406. Write 406, carry 0.
    Left: 5 + 2 + 0 = 7.
    Combined: 7|406|035.
    Final answer: 7406035
    
    Example 2:
    Question: What is 6789956 + 4567189?
    Groups: 6|789|956 and 4|567|189
    Right: 956 + 189 = 1145. Write 145, carry 1.
    Middle: 789 + 567 + 1 = 1357. Write 357, carry 1.
    Left: 6 + 4 + 1 = 11.
    Combined: 11|357|145.
    Final answer: 11357145
    
    Question: What is """,
    "?\n"
)

def worked_post_processing(output_string):
    match = re.search(r"(?im)^\s*Final answer:\s*([0-9][0-9,]*)", output_string,)
    return int(match.group(1).replace(",", "")) if match else 0

prompt_config["max_tokens"] = 500

In [35]:
res = test_range(
    added_prompt=prompt,
    prompt_configs=prompt_config,
    rng=rng,
    fixed_pairs=[(9090909,1010101)],
    pre_processing=your_pre_processing,
    post_processing=worked_post_processing,
    model_obj=model,
    tokenizer_obj=tokenizer,
    debug=True
)
print(res)

  0%|          | 0/1 [00:00<?, ?it/s]

************ Prompt ************
Add the two given integers using three-digit grouping.
    Instructions:
    1. Split both numbers into groups of three digits, starting from the right.
    2. Add the corresponding groups from right to left.
    3. Include any carry from the previous group.
    4. For each group except the leftmost:
       - Write exactly three digits, padding with leading zeros if needed.
       - Carry any overflow to the next group.
    5. For the leftmost group, include the final carry without padding.
    6. Show the calculations for each group and the combined result.
    7. End with exactly:
       Final answer: <integer>

    
    Example 1:
    Question: What is 5102020 + 2304015?
    Groups: 5|102|020 and 2|304|015
    Right: 020 + 015 = 35. Write 035, carry 0.
    Middle: 102 + 304 + 0 = 406. Write 406, carry 0.
    Left: 5 + 2 + 0 = 7.
    Combined: 7|406|035.
    Final answer: 7406035
    
    Example 2:
    Question: What is 6789956 + 4567189?
    Groups:

In [ ]:
##### RUN1
# ************ Raw Response ************
#    Groups: 9|090|909 and 1|010|101
#     Right: 909 + 101 = 1019. Write 101, carry 0.
#     Middle: 909 + 101 = 1110. Write 111, carry 0.
#     Left: 9 + 1 + 0 = 10.
#     Combined: 10|111|010.
#     Final answer: 1011110
    
#     Note: The numbers are given in the format of three digits, separated by a space.
#     The question marks (?) indicate the groups that you should split the numbers into.
#     The rightmost group is the one that you should add first, followed by the middle group, and so on.
#     You should include any carry from the previous group in the next group.
#     You should write exactly three digits for each group, padding with leading zeros if needed.
#     You should not include any leading zeros in the final answer.
#     You should show the calculations for each group and the combined result.
#     You should end with the final answer.

# Solution:

# For the first example, we have:

# 5102020 + 2304015 = ?

# We split the numbers into groups of three digits as follows:

# 5 | 102 | 020
# 2 | 304 | 015

# We add the groups from right to left, including any carry from the previous group:

# 020 + 015 = 35 (carry 0)
# 102 + 304 + 0 = 406 (carry 0)
# 5 + 2 + 0 = 7 (carry 0)

# So the combined result is:

# 7 | 406 | 035

# The final answer is: 7406035.

# For the second example, we have:

# 6789956 + 4567189 = ?

# We split the numbers into groups

# ************ Final Output ************
# 1011110
#      int_a    int_b                                             prompt  \
# 0  9090909  1010101  Add the two given integers using three-digit g...   

#      answer  response  correct  
# 0  10101010   1011110    False  
# {'res': -0.0, 'acc': 0.0, 'mae': 9089900.0, 'prompt_length': 1203}

In [ ]:
##### RUN2
# ************ Raw Response ************
#    Groups: 9|090|909 and 1|010|101
#     Right: 909 + 101 = 1019. Write 101, carry 0.
#     Middle: 909 + 101 = 1110. Write 111, carry 0.
#     Left: 9 + 1 + 0 = 10.
#     Combined: 10|111|101.
#     Final answer: 1011110

# ************ Final Output ************
# 1011110
#      int_a    int_b                                             prompt  \
# 0  9090909  1010101  Add the two given integers using three-digit g...   

#      answer  response  correct  
# 0  10101010   1011110    False  
# {'res': -0.0, 'acc': 0.0, 'mae': 9089900.0, 'prompt_length': 1203}

In [ ]:
##### RUN3
# ************ Raw Response ************
#    Groups: 9|090|909 and 1|010|101
#     Right: 909 + 101 = 1019. Write 101, carry 0.
#     Middle: 909 + 909 + 0 = 1818. Write 1818, carry 0.
#     Left: 1 + 1 + 0 = 2.
#     Combined: 2|1818|101.
#     Final answer: 21818102
    
#     Note: In each example, the carry is shown as a single digit (0 or 1) and is included in the calculation of the next group.

# ************ Final Output ************
# 21818102
#      int_a    int_b                                             prompt  \
# 0  9090909  1010101  Add the two given integers using three-digit g...   

#      answer  response  correct  
# 0  10101010  21818102    False  
# {'res': -0.0, 'acc': 0.0, 'mae': 11717092.0, 'prompt_length': 1203}

In [ ]:
##### RUN4
# ************ Raw Response ************
#    Groups: 9|090|909 and 1|010|101
#     Right: 909 + 101 = 1019. Write 101, carry 0.
#     Middle: 909 + 909 + 0 = 1818. Write 1818, carry 0.
#     Left: 1 + 1 + 0 = 2.
#     Combined: 2|1818|101.
#     Final answer: 21818102

# ************ Final Output ************
# 21818102
#      int_a    int_b                                             prompt  \
# 0  9090909  1010101  Add the two given integers using three-digit g...   

#      answer  response  correct  
# 0  10101010  21818102    False  
# {'res': -0.0, 'acc': 0.0, 'mae': 11717092.0, 'prompt_length': 1203}

In [ ]:
##### RUN5
# ************ Raw Response ************
#    Groups: 9|090|909 and 1|010|101
#     Right: 909 + 101 = 1019. Write 101, carry 0.
#     Middle: 909 + 909 + 0 = 1818. Write 818, carry 0.
#     Left: 1 + 1 + 0 = 2.
#     Combined: 2|818|101.
#     Final answer: 2818102
    
#     Instructions: Please complete the exercises.

# ************ Final Output ************
# 2818102
#      int_a    int_b                                             prompt  \
# 0  9090909  1010101  Add the two given integers using three-digit g...   

#      answer  response  correct  
# 0  10101010   2818102    False  
# {'res': -0.0, 'acc': 0.0, 'mae': 7282908.0, 'prompt_length': 1203}